# CART_databricks

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.fpm import FPGrowth

In [0]:
file_path = "/Volumes/workspace/ccbd_cart/ccbd_cart_data/online_retail_clean.csv"

print("📥 Loading CSV file from Managed Volume...")
df_spark = spark.read.csv(
    file_path, 
    header=True, 
    inferSchema=True
)

print(f"✅ Rows successfully loaded: {df_spark.count()}")

print("⚙️ Converting to Delta Lake format (ACID Transactions & Optimization)...")

# Save the DataFrame as a governed Delta Table (Silver/Gold Layer)
# Format: catalog.schema.table
table_name = "workspace.ccbd_cart.online_retail_delta"

df_spark.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"🎉 Success! The table '{table_name}' is ready and optimized in the Lakehouse.")

# Read the governed table back for the analysis
df_delta = spark.read.table(table_name)

📥 Loading CSV file from Managed Volume...
✅ Rows successfully loaded: 530104
⚙️ Converting to Delta Lake format (ACID Transactions & Optimization)...
🎉 Success! The table 'workspace.ccbd_cart.online_retail_delta' is ready and optimized in the Lakehouse.


In [0]:
table_name = "workspace.ccbd_cart.online_retail_delta"
df_delta = spark.read.table(table_name)

## Query

### Query 1

In [0]:
spark.sql(f"""
    SELECT 
        YEAR(InvoiceDate) AS year,
        MONTH(InvoiceDate) AS month,
        COUNT(DISTINCT InvoiceNo) AS total_orders,
        ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue
    FROM {table_name}
    GROUP BY year, month
    ORDER BY year, month
""").show(10)

+----+-----+------------+-------------+
|year|month|total_orders|total_revenue|
+----+-----+------------+-------------+
|2010|   12|        1559|    823746.14|
|2011|    1|        1086|    691364.56|
|2011|    2|        1100|    523631.89|
|2011|    3|        1454|    717639.36|
|2011|    4|        1246|    537808.62|
|2011|    5|        1681|    770536.02|
|2011|    6|        1533|     761739.9|
|2011|    7|        1475|    719221.19|
|2011|    8|        1361|    759138.38|
|2011|    9|        1837|   1058590.17|
+----+-----+------------+-------------+
only showing top 10 rows


In [0]:
df_delta.groupBy(
    F.year("InvoiceDate").alias("year"),
    F.month("InvoiceDate").alias("month")
).agg(
    F.countDistinct("InvoiceNo").alias("total_orders"),
    F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("total_revenue")
).orderBy("year", "month").show(10)

+----+-----+------------+-------------+
|year|month|total_orders|total_revenue|
+----+-----+------------+-------------+
|2010|   12|        1559|    823746.14|
|2011|    1|        1086|    691364.56|
|2011|    2|        1100|    523631.89|
|2011|    3|        1454|    717639.36|
|2011|    4|        1246|    537808.62|
|2011|    5|        1681|    770536.02|
|2011|    6|        1533|     761739.9|
|2011|    7|        1475|    719221.19|
|2011|    8|        1361|    759138.38|
|2011|    9|        1837|   1058590.17|
+----+-----+------------+-------------+
only showing top 10 rows


### Query 2

In [0]:
spark.sql(f"""
    SELECT 
        Country AS country,
        COUNT(DISTINCT InvoiceNo) AS total_orders,
        ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue,
        ROUND(SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo), 2) AS average_order_value
    FROM {table_name}
    GROUP BY Country
    ORDER BY total_revenue DESC
""").show(10)

+--------------+------------+-------------+-------------------+
|       country|total_orders|total_revenue|average_order_value|
+--------------+------------+-------------+-------------------+
|United Kingdom|       18019|   9025222.08|             500.87|
|   Netherlands|          94|    285446.34|            3036.66|
|          EIRE|         288|    283453.96|             984.22|
|       Germany|         457|    228867.14|              500.8|
|        France|         392|    209715.11|             534.99|
|     Australia|          57|    138521.31|             2430.2|
|         Spain|          90|     61577.11|             684.19|
|   Switzerland|          54|      57089.9|            1057.22|
|       Belgium|          98|     41196.34|             420.37|
|        Sweden|          36|     38378.33|            1066.06|
+--------------+------------+-------------+-------------------+
only showing top 10 rows


In [0]:
df_delta.groupBy("Country").agg(
    F.countDistinct("InvoiceNo").alias("total_orders"),
    F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("total_revenue"),
    F.round(
        F.sum(F.col("Quantity") * F.col("UnitPrice")) / F.countDistinct("InvoiceNo"), 2
    ).alias("average_order_value")
).withColumnRenamed("Country", "country") \
 .orderBy(F.col("total_revenue").desc()).show(10)

+--------------+------------+-------------+-------------------+
|       country|total_orders|total_revenue|average_order_value|
+--------------+------------+-------------+-------------------+
|United Kingdom|       18019|   9025222.08|             500.87|
|   Netherlands|          94|    285446.34|            3036.66|
|          EIRE|         288|    283453.96|             984.22|
|       Germany|         457|    228867.14|              500.8|
|        France|         392|    209715.11|             534.99|
|     Australia|          57|    138521.31|             2430.2|
|         Spain|          90|     61577.11|             684.19|
|   Switzerland|          54|      57089.9|            1057.22|
|       Belgium|          98|     41196.34|             420.37|
|        Sweden|          36|     38378.33|            1066.06|
+--------------+------------+-------------+-------------------+
only showing top 10 rows


### Query 3

In [0]:
spark.sql(f"""
    SELECT 
        date_format(InvoiceDate, 'EEEE') AS day_of_week,
        CASE 
            WHEN hour(InvoiceDate) BETWEEN 6 AND 12 THEN '1 - Morning (06:00 am - 12:59 pm)'
            WHEN hour(InvoiceDate) BETWEEN 13 AND 17 THEN '2 - Afternoon (01:00 pm - 05:59 pm)'
            WHEN hour(InvoiceDate) BETWEEN 18 AND 22 THEN '3 - Evening (06:00 pm - 10:59 pm)'
            ELSE '4 - Night (11:00 pm - 05:59 am)'
        END AS time_bracket,
        COUNT(DISTINCT InvoiceNo) AS total_orders
    FROM {table_name}
    GROUP BY day_of_week, time_bracket
    ORDER BY total_orders DESC
""").show(10)

+-----------+--------------------+------------+
|day_of_week|        time_bracket|total_orders|
+-----------+--------------------+------------+
|   Thursday|2 - Afternoon (01...|        2032|
|  Wednesday|1 - Morning (06:0...|        1932|
|   Thursday|1 - Morning (06:0...|        1884|
|    Tuesday|1 - Morning (06:0...|        1838|
|  Wednesday|2 - Afternoon (01...|        1755|
|     Friday|1 - Morning (06:0...|        1750|
|    Tuesday|2 - Afternoon (01...|        1707|
|     Monday|1 - Morning (06:0...|        1574|
|     Monday|2 - Afternoon (01...|        1550|
|     Friday|2 - Afternoon (01...|        1378|
+-----------+--------------------+------------+
only showing top 10 rows


In [0]:
df_delta.withColumn("day_of_week", F.date_format("InvoiceDate", "EEEE")) \
    .withColumn("time_bracket", 
        F.when((F.hour("InvoiceDate") >= 6) & (F.hour("InvoiceDate") <= 12), '1 - Morning (06:00 am - 12:59 pm)')
         .when((F.hour("InvoiceDate") >= 13) & (F.hour("InvoiceDate") <= 17), '2 - Afternoon (01:00 pm - 05:59 pm)')
         .when((F.hour("InvoiceDate") >= 18) & (F.hour("InvoiceDate") <= 22), '3 - Evening (06:00 pm - 10:59 pm)')
         .otherwise('4 - Night (11:00 pm - 05:59 am)')
    ) \
    .groupBy("day_of_week", "time_bracket") \
    .agg(F.countDistinct("InvoiceNo").alias("total_orders")) \
    .orderBy(F.col("total_orders").desc()).show(10)

+-----------+--------------------+------------+
|day_of_week|        time_bracket|total_orders|
+-----------+--------------------+------------+
|   Thursday|2 - Afternoon (01...|        2032|
|  Wednesday|1 - Morning (06:0...|        1932|
|   Thursday|1 - Morning (06:0...|        1884|
|    Tuesday|1 - Morning (06:0...|        1838|
|  Wednesday|2 - Afternoon (01...|        1755|
|     Friday|1 - Morning (06:0...|        1750|
|    Tuesday|2 - Afternoon (01...|        1707|
|     Monday|1 - Morning (06:0...|        1574|
|     Monday|2 - Afternoon (01...|        1550|
|     Friday|2 - Afternoon (01...|        1378|
+-----------+--------------------+------------+
only showing top 10 rows


### Query 4

In [0]:
spark.sql(f"""
    WITH reference_date AS (
        SELECT MAX(to_date(InvoiceDate)) AS max_date FROM {table_name}
    ),
    rfm_base AS (
        SELECT 
            CustomerID,
            datediff((SELECT max_date FROM reference_date), MAX(to_date(InvoiceDate))) AS Recency,
            COUNT(DISTINCT InvoiceNo) AS Frequency,
            ROUND(SUM(Quantity * UnitPrice), 2) AS Monetary
        FROM {table_name}
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
        GROUP BY CustomerID
    )
    SELECT * FROM rfm_base
    ORDER BY Monetary DESC, Frequency DESC
""").show(10)

+----------+-------+---------+---------+
|CustomerID|Recency|Frequency| Monetary|
+----------+-------+---------+---------+
|     14646|      1|       73|280206.02|
|     18102|      0|       60| 259657.3|
|     17450|      8|       46|194550.79|
|     16446|      0|        2| 168472.5|
|     14911|      1|      201|143825.06|
|     12415|     24|       21|124914.53|
|     14156|      9|       55|117379.63|
|     17511|      2|       31| 91062.38|
|     16029|     38|       63| 81024.84|
|     12346|    325|        1|  77183.6|
+----------+-------+---------+---------+
only showing top 10 rows


In [0]:
max_date_val = df_delta.select(F.max(F.to_date("InvoiceDate"))).collect()[0][0]

df_delta.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(
        F.datediff(F.lit(max_date_val), F.max(F.to_date("InvoiceDate"))).alias("Recency"),
        F.countDistinct("InvoiceNo").alias("Frequency"),
        F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).alias("Monetary")
    ).orderBy(F.col("Monetary").desc(), F.col("Frequency").desc()).show(10)

+----------+-------+---------+---------+
|CustomerID|Recency|Frequency| Monetary|
+----------+-------+---------+---------+
|     14646|      1|       73|280206.02|
|     18102|      0|       60| 259657.3|
|     17450|      8|       46|194550.79|
|     16446|      0|        2| 168472.5|
|     14911|      1|      201|143825.06|
|     12415|     24|       21|124914.53|
|     14156|      9|       55|117379.63|
|     17511|      2|       31| 91062.38|
|     16029|     38|       63| 81024.84|
|     12346|    325|        1|  77183.6|
+----------+-------+---------+---------+
only showing top 10 rows


#### Query 5

In [0]:
spark.sql(f"""
    WITH customer_cohort AS (
        SELECT 
            CustomerID,
            date_trunc('month', MIN(InvoiceDate)) AS cohort_month
        FROM {table_name}
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
        GROUP BY CustomerID
    ),
    customer_purchases AS (
        SELECT DISTINCT 
            CustomerID,
            date_trunc('month', InvoiceDate) AS purchase_month
        FROM {table_name}
        WHERE CustomerID IS NOT NULL AND CustomerID != 0
    )
    SELECT 
        date_format(cc.cohort_month, 'yyyy-MM') AS acquisition_month,
        CAST(months_between(cp.purchase_month, cc.cohort_month) AS INT) AS retention_month_index,
        COUNT(DISTINCT cp.CustomerID) AS active_customers
    FROM customer_cohort cc
    INNER JOIN customer_purchases cp ON cc.CustomerID = cp.CustomerID
    GROUP BY acquisition_month, retention_month_index
    ORDER BY acquisition_month ASC, retention_month_index ASC
""").show(10)

+-----------------+---------------------+----------------+
|acquisition_month|retention_month_index|active_customers|
+-----------------+---------------------+----------------+
|          2010-12|                    0|             885|
|          2010-12|                    1|             324|
|          2010-12|                    2|             286|
|          2010-12|                    3|             340|
|          2010-12|                    4|             321|
|          2010-12|                    5|             352|
|          2010-12|                    6|             321|
|          2010-12|                    7|             309|
|          2010-12|                    8|             313|
|          2010-12|                    9|             350|
+-----------------+---------------------+----------------+
only showing top 10 rows


In [0]:
cohort_df = df_delta.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(F.date_trunc("month", F.min("InvoiceDate")).alias("cohort_month"))

purchases_df = df_delta.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .select("CustomerID", F.date_trunc("month", "InvoiceDate").alias("purchase_month")).distinct()

cohort_df.join(purchases_df, "CustomerID") \
    .withColumn("acquisition_month", F.date_format("cohort_month", "yyyy-MM")) \
    .withColumn("retention_month_index", F.months_between(F.col("purchase_month"), F.col("cohort_month")).cast("int")) \
    .groupBy("acquisition_month", "retention_month_index") \
    .agg(F.countDistinct("CustomerID").alias("active_customers")) \
    .orderBy("acquisition_month", "retention_month_index").show(10)

+-----------------+---------------------+----------------+
|acquisition_month|retention_month_index|active_customers|
+-----------------+---------------------+----------------+
|          2010-12|                    0|             885|
|          2010-12|                    1|             324|
|          2010-12|                    2|             286|
|          2010-12|                    3|             340|
|          2010-12|                    4|             321|
|          2010-12|                    5|             352|
|          2010-12|                    6|             321|
|          2010-12|                    7|             309|
|          2010-12|                    8|             313|
|          2010-12|                    9|             350|
+-----------------+---------------------+----------------+
only showing top 10 rows


### Query 6

In [0]:
spark.sql(f"""
    WITH product_revenue AS (
        SELECT 
            StockCode,
            Description AS product_name,
            SUM(Quantity * UnitPrice) AS total_revenue
        FROM {table_name}
        GROUP BY StockCode, Description
    ),
    cumulative_data AS (
        SELECT 
            product_name,
            total_revenue,
            SUM(total_revenue) OVER(ORDER BY total_revenue DESC) AS cumulative_revenue,
            SUM(total_revenue) OVER() AS global_revenue
        FROM product_revenue
    )
    SELECT 
        product_name,
        ROUND(total_revenue, 2) AS total_revenue,
        ROUND(cumulative_revenue, 2) AS cumulative_revenue,
        ROUND((cumulative_revenue / global_revenue) * 100, 2) AS cumulative_percentage
    FROM cumulative_data
    ORDER BY total_revenue DESC
    LIMIT 50
""").show(10)

+--------------------+-------------+------------------+---------------------+
|        product_name|total_revenue|cumulative_revenue|cumulative_percentage|
+--------------------+-------------+------------------+---------------------+
|      DOTCOM POSTAGE|    206248.77|         206248.77|                 1.93|
|REGENCY CAKESTAND...|    174484.74|         380733.51|                 3.57|
|PAPER CRAFT , LIT...|     168469.6|         549203.11|                 5.15|
|WHITE HANGING HEA...|    104340.29|          653543.4|                 6.13|
|       PARTY BUNTING|     99504.33|         753047.73|                 7.06|
|JUMBO BAG RED RET...|     94340.05|         847387.78|                 7.94|
|MEDIUM CERAMIC TO...|     81700.92|          929088.7|                 8.71|
|              Manual|     78110.27|        1007198.97|                 9.44|
|             POSTAGE|     78101.88|        1085300.85|                10.17|
|  RABBIT NIGHT LIGHT|     66964.99|        1152265.84|         

In [0]:
prod_rev_df = df_delta.groupBy("StockCode", "Description") \
    .agg(F.sum(F.col("Quantity") * F.col("UnitPrice")).alias("total_revenue"))

window_spec_cum = Window.orderBy(F.col("total_revenue").desc())
window_spec_glob = Window.partitionBy() 

prod_rev_df \
    .withColumn("cumulative_revenue", F.sum("total_revenue").over(window_spec_cum)) \
    .withColumn("global_revenue", F.sum("total_revenue").over(window_spec_glob)) \
    .withColumn("cumulative_percentage", F.round((F.col("cumulative_revenue") / F.col("global_revenue")) * 100, 2)) \
    .select(
        F.col("Description").alias("product_name"),
        F.round("total_revenue", 2).alias("total_revenue"),
        F.round("cumulative_revenue", 2).alias("cumulative_revenue"),
        "cumulative_percentage"
    ).orderBy(F.col("total_revenue").desc()).limit(10).show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------------+-------------+------------------+---------------------+
|        product_name|total_revenue|cumulative_revenue|cumulative_percentage|
+--------------------+-------------+------------------+---------------------+
|      DOTCOM POSTAGE|    206248.77|         206248.77|                 1.93|
|REGENCY CAKESTAND...|    174484.74|         380733.51|                 3.57|
|PAPER CRAFT , LIT...|     168469.6|         549203.11|                 5.15|
|WHITE HANGING HEA...|    104340.29|          653543.4|                 6.13|
|       PARTY BUNTING|     99504.33|         753047.73|                 7.06|
|JUMBO BAG RED RET...|     94340.05|         847387.78|                 7.94|
|MEDIUM CERAMIC TO...|     81700.92|          929088.7|                 8.71|
|              Manual|     78110.27|        1007198.97|                 9.44|
|             POSTAGE|     78101.88|        1085300.85|                10.17|
|  RABBIT NIGHT LIGHT|     66964.99|        1152265.84|         

## K-means

In [0]:
# Re-read the table from Unity Catalog to ensure fresh data
table_name = "workspace.ccbd_cart.online_retail_delta"
df_delta = spark.read.table(table_name)

print("⚙️ Calculating base RFM metrics...")
max_date_val = df_delta.select(F.max(F.to_date("InvoiceDate"))).collect()[0][0]

rfm_df = df_delta.filter((F.col("CustomerID").isNotNull()) & (F.col("CustomerID") != 0)) \
    .groupBy("CustomerID") \
    .agg(
        F.datediff(F.lit(max_date_val), F.max(F.to_date("InvoiceDate"))).cast("double").alias("Recency"),
        F.countDistinct("InvoiceNo").cast("double").alias("Frequency"),
        F.round(F.sum(F.col("Quantity") * F.col("UnitPrice")), 2).cast("double").alias("Monetary")
    )

print(f"✅ RFM Data Ready. Unique customers to segment: {rfm_df.count()}")

print("⚙️ Feature Engineering (Assembler & Scaler)...")
assembler = VectorAssembler(
    inputCols=["Recency", "Frequency", "Monetary"],
    outputCol="raw_features"
)
rfm_assembled = assembler.transform(rfm_df)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)
scaler_model = scaler.fit(rfm_assembled)
rfm_scaled = scaler_model.transform(rfm_assembled)

print("🧠 Training K-Means model (k=3)...")
kmeans = KMeans(
    featuresCol="scaled_features",
    predictionCol="Cluster_Segment",
    k=3,
    seed=42
)
kmeans_model = kmeans.fit(rfm_scaled)

predictions = kmeans_model.transform(rfm_scaled)

print("🧪 Clustering Quality Evaluation...")
evaluator = ClusteringEvaluator(
    predictionCol="Cluster_Segment",
    featuresCol="scaled_features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
silhouette_score = evaluator.evaluate(predictions)
print(f"✅ Model Trained! Silhouette Score: {silhouette_score:.4f}")

print("📊 Cluster Profiling (Business View)...")
cluster_profile = predictions.groupBy("Cluster_Segment").agg(
    F.count("CustomerID").alias("customer_count"),
    F.round(F.avg("Recency"), 0).alias("avg_recency_days"),
    F.round(F.avg("Frequency"), 1).alias("avg_frequency_orders"),
    F.round(F.avg("Monetary"), 2).alias("avg_monetary_gbp")
).orderBy("Cluster_Segment")

cluster_profile.show()

⚙️ Calculating base RFM metrics...
✅ RFM Data Ready. Unique customers to segment: 4338
⚙️ Feature Engineering (Assembler & Scaler)...
🧠 Training K-Means model (k=3)...
🧪 Clustering Quality Evaluation...
✅ Model Trained! Silhouette Score: 0.7133
📊 Cluster Profiling (Business View)...
+---------------+--------------+----------------+--------------------+----------------+
|Cluster_Segment|customer_count|avg_recency_days|avg_frequency_orders|avg_monetary_gbp|
+---------------+--------------+----------------+--------------------+----------------+
|              0|          3233|            41.0|                 4.9|         2011.56|
|              1|          1091|           246.0|                 1.6|          630.24|
|              2|            14|             6.0|                80.2|       122888.41|
+---------------+--------------+----------------+--------------------+----------------+



## FP-Growth

In [0]:
print("⚙️ Grouping items by invoice into transaction baskets...")
basket_df = df_delta.filter(~F.col("InvoiceNo").startswith("C")) \
    .filter(F.col("Description").isNotNull()) \
    .groupBy("InvoiceNo") \
    .agg(F.collect_set("Description").alias("items"))

print(f"✅ Unique transactions ready for FP-Growth: {basket_df.count()}")

print("🧠 Extracting frequent patterns and training FP-Growth...")
fp_growth = FPGrowth(
    itemsCol="items",
    minSupport=0.02,
    minConfidence=0.3
)
model = fp_growth.fit(basket_df)
print("✅ Algorithm completed successfully!")

print("📊 Top 10 Frequent Itemsets:")
model.freqItemsets.orderBy(F.col("freq").desc()).show(10, truncate=False)

print("🔮 Top 10 Predictive Association Rules (by Lift):")
association_rules = model.associationRules.orderBy(F.col("lift").desc())
association_rules.show(10, truncate=False)

print("🚀 Exporting Association Rules to Delta Table...")
# Save the rules back to the Lakehouse so they can be queried by BI tools
rules_table_name = "workspace.ccbd_cart.association_rules"

association_rules.write.format("delta").mode("overwrite").saveAsTable(rules_table_name)

print(f"🎉 Export completed! The recommendation engine rules are saved in '{rules_table_name}'.")

⚙️ Grouping items by invoice into transaction baskets...
✅ Unique transactions ready for FP-Growth: 19960
🧠 Extracting frequent patterns and training FP-Growth...
✅ Algorithm completed successfully!
📊 Top 10 Frequent Itemsets:
+------------------------------------+----+
|items                               |freq|
+------------------------------------+----+
|[WHITE HANGING HEART T-LIGHT HOLDER]|2256|
|[JUMBO BAG RED RETROSPOT]           |2089|
|[REGENCY CAKESTAND 3 TIER]          |1988|
|[PARTY BUNTING]                     |1685|
|[LUNCH BAG RED RETROSPOT]           |1564|
|[ASSORTED COLOUR BIRD ORNAMENT]     |1455|
|[SET OF 3 CAKE TINS PANTRY DESIGN ] |1385|
|[PACK OF 72 RETROSPOT CAKE CASES]   |1320|
|[LUNCH BAG  BLACK SKULL.]           |1273|
|[NATURAL SLATE HEART CHALKBOARD ]   |1249|
+------------------------------------+----+
only showing top 10 rows
🔮 Top 10 Predictive Association Rules (by Lift):
+-------------------------------------------------------------------+--------------